# Copy of fusion-probe.ipynb

Audio-Flamingo-3 layer probes and audio-language fusion-weight experiments.

Set the metadata, representation, and prediction-output paths for the Flamingo run, then run the selected probe section.

Use the setup steps in the repository README before running this notebook.


In [ ]:
!pip install iterative-stratification


In [ ]:
# =========================================================
# MULTILABEL SUBTLE EMOTION PROBE
# CACHED VERSION (FAST)
# =========================================================

import os
import ast
import json
import glob
import shutil

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import f1_score
from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

REP_ROOT = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo"
)

LOCAL_REP_ROOT = "/content/reps_local_flamingo"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

THRESHOLD = 0.15

MIN_FREQ = 20

# =========================================================
# COPY ONLY PT FILES
# =========================================================

os.makedirs(
    LOCAL_REP_ROOT,
    exist_ok=True
)

pt_files = glob.glob(
    os.path.join(REP_ROOT, "*.pt")
)

print(f"Found {len(pt_files)} pt files")

for src_path in tqdm(pt_files):

    filename = os.path.basename(src_path)

    dst_path = os.path.join(
        LOCAL_REP_ROOT,
        filename
    )

    if not os.path.exists(dst_path):

        shutil.copy2(
            src_path,
            dst_path
        )

print("Finished copying pt files locally.")

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

# Keep only required columns
df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# REMOVE MISSING REPS
# =========================================================

df = df[
    df["id"].apply(
        lambda x:
        os.path.exists(
            os.path.join(
                LOCAL_REP_ROOT,
                f"{x}.pt"
            )
        )
    )
].reset_index(drop=True)

print(f"Remaining samples: {len(df)}")

# =========================================================
# BUILD EMOTION VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        # ---------------------------------------------
        # REMOVE UNSURE
        # ---------------------------------------------

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

# ---------------------------------------------
# FILTER RARE EMOTIONS
# ---------------------------------------------

EMOTIONS = sorted([

    emo

    for emo, freq
    in emotion_counter.items()

    if freq >= MIN_FREQ
])

EMO2IDX = {
    emo: i
    for i, emo in enumerate(EMOTIONS)
}

IDX2EMO = {
    i: emo
    for emo, i in EMO2IDX.items()
}

NUM_LABELS = len(EMOTIONS)

print(f"\nDetected {NUM_LABELS} emotions:\n")
print(EMOTIONS)

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []
valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    # ---------------------------------------------
    # REMOVE UNSURE
    # ---------------------------------------------

    counts = {

        k: v

        for k, v
        in counts.items()

        if k != "unsure"
    }

    # ---------------------------------------------
    # SKIP EMPTY
    # ---------------------------------------------

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

# ---------------------------------------------
# FILTER DATAFRAME
# ---------------------------------------------

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

print("\nTargets shape:")
print(targets.shape)

# =========================================================
# STRATIFIED SPLIT
# =========================================================

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(

    msss.split(
        np.zeros(len(targets)),
        (targets > 0).astype(int)
    )
)

temp_targets = targets[temp_idx]

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_subidx, test_subidx = next(

    msss2.split(
        np.zeros(len(temp_targets)),
        (temp_targets > 0).astype(int)
    )
)

val_idx = temp_idx[val_subidx]
test_idx = temp_idx[test_subidx]

print("\nSplit sizes:")
print(
    len(train_idx),
    len(val_idx),
    len(test_idx)
)

# =========================================================
# SAVE SPLITS
# =========================================================

split_dict = {

    "train": train_idx.tolist(),
    "val": val_idx.tolist(),
    "test": test_idx.tolist()
}

with open(
    "/content/splits.json",
    "w"
) as f:

    json.dump(split_dict, f)

print("\nSaved splits.")

# =========================================================
# AUTO DETECT LAYERS
# =========================================================

sample_rep = torch.load(
    os.path.join(
        LOCAL_REP_ROOT,
        "0.pt"
    ),
    map_location="cpu"
)

ALL_LAYERS = sorted(
    sample_rep.keys()
)

print("\nDetected layers:")
print(ALL_LAYERS)

# =========================================================
# CACHE REPRESENTATIONS
# =========================================================

def process_rep(x):

    x = x.float()

    # ---------------------------------------------
    # POOLING
    # ---------------------------------------------

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    # ---------------------------------------------
    # NORMALIZE
    # ---------------------------------------------

    x = (
        x - x.mean()
    ) / (
        x.std() + 1e-6
    )

    return x

# =========================================================
# DATASET
# =========================================================

class CachedDataset(Dataset):

    def __init__(
        self,
        X,
        Y
    ):

        self.X = X
        self.Y = Y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.Y[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# EVALUATION
# =========================================================


def evaluate(
    model,
    loader
):

    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            preds = (
                probs > THRESHOLD
            ).float()

            all_probs.append(
                probs.cpu().numpy()
            )

            all_preds.append(
                preds.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

    all_probs = np.concatenate(all_probs)

    all_preds = np.concatenate(all_preds)

    all_labels = np.concatenate(all_labels)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    return (
        macro_f1,
        micro_f1,
        all_probs,
        all_preds,
        all_labels
    )

# =========================================================
# MAIN LOOP
# =========================================================

results = []

# ---------------------------------------------
# START WITH IMPORTANT LAYERS
# ---------------------------------------------

SELECTED_LAYERS = [

    "audio_32",

    "projector",

    "lm_13",

    "lm_27"
]

for layer_key in SELECTED_LAYERS:

    print("\n===================================")
    print(layer_key)
    print("===================================")

    # =====================================================
    # CACHE LAYER
    # =====================================================

    cached_X = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        rep_id = row["id"]

        rep_path = os.path.join(
            LOCAL_REP_ROOT,
            f"{rep_id}.pt"
        )

        rep = torch.load(
            rep_path,
            map_location="cpu"
        )

        x = process_rep(
            rep[layer_key]
        )

        cached_X.append(x)

    cached_X = torch.stack(cached_X)

    cached_Y = torch.tensor(
        targets,
        dtype=torch.float
    )

    print("\nCached shape:")
    print(cached_X.shape)

    # =====================================================
    # SPLIT
    # =====================================================

    train_dataset = CachedDataset(

        cached_X[train_idx],
        cached_Y[train_idx]
    )

    val_dataset = CachedDataset(

        cached_X[val_idx],
        cached_Y[val_idx]
    )

    test_dataset = CachedDataset(

        cached_X[test_idx],
        cached_Y[test_idx]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # =====================================================
    # MODEL
    # =====================================================

    input_dim = cached_X.shape[-1]

    model = LinearProbe(
        input_dim,
        NUM_LABELS
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LR
    )

    best_val = -1

    # =====================================================
    # TRAIN
    # =====================================================

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        (
            val_macro,
            val_micro,
            _,
            _,
            _
        )  = evaluate(
            model,
            val_loader
        )

        print(
            f"Epoch {epoch} | "
            f"Loss {total_loss:.4f} | "
            f"Val Macro {val_macro:.4f} | "
            f"Val Micro {val_micro:.4f}"
        )

        if val_macro > best_val:

            best_val = val_macro

            torch.save(
                model.state_dict(),
                f"/content/{layer_key}_best.pt"
            )

    # =====================================================
    # TEST
    # =====================================================

    model.load_state_dict(
        torch.load(
            f"/content/{layer_key}_best.pt"
        )
    )

    (
    test_macro,
    test_micro,
    all_probs,
    all_preds,
    all_labels
    ) = evaluate(
        model,
        test_loader
    )
    # =====================================================
    # SAVE TEST PREDICTIONS
    # =====================================================

    os.makedirs(
        "/content/drive/MyDrive/evaluation_predictions/flamingo",
        exist_ok=True
    )

    eval_df = pd.DataFrame({

        "sample_index": test_idx,

        "sample_id": df.iloc[test_idx]["id"].values,

        "label_counts": df.iloc[test_idx]["label_counts"].values

    })

    for i, emo in enumerate(EMOTIONS):

        eval_df[f"{emo}_gold"] = all_labels[:, i]

        eval_df[f"{emo}_pred"] = all_preds[:, i]

        eval_df[f"{emo}_prob"] = all_probs[:, i]

    print("\nTEST RESULTS")
    print(
        f"Macro F1: {test_macro:.4f}"
    )
    print(
        f"Micro F1: {test_micro:.4f}"
    )

    results.append({

        "layer": layer_key,

        "macro_f1": test_macro,

        "micro_f1": test_micro
    })
    print(layer_key)
    eval_df.to_csv(

    f"/content/drive/MyDrive/evaluation_predictions/flamingo/{layer_key}.csv",

    index=False
)

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "/content/subtle_probe_results.csv",
    index=False
)

print("\nFINAL RESULTS")
print(results_df)

In [ ]:

# =========================================================
# FUSION PROBE
# LIGHTWEIGHT SPEECH AMPLIFICATION
# =========================================================

import os
import ast
import json

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import f1_score

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_qwen/task3_results.csv"
)

REP_ROOT = "/content/reps_local_flamingo"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

THRESHOLD = 0.15

# =========================================================
# LAYERS
# =========================================================

# ---------------------------------------------
# strongest subtle emotion layer
# ---------------------------------------------

LLM_LAYER = "lm_15"

# ---------------------------------------------
# strongest audio layer
# ---------------------------------------------

AUDIO_LAYER = "projector"

# =========================================================
# FUSION WEIGHTS
# =========================================================

# audio amplification weights

ALPHAS = [

    0.0,   # LLM only baseline

    0.25,

    0.50,

    0.75,

    1.0    # audio only baseline

]

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# LOAD SPLITS
# =========================================================

with open("/content/splits.json") as f:

    split_dict = json.load(f)

train_idx = split_dict["train"]
val_idx = split_dict["val"]
test_idx = split_dict["test"]

# =========================================================
# BUILD VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

EMOTIONS = sorted([
    emo
    for emo, freq
    in emotion_counter.items()
    if freq >= 20
])

EMO2IDX = {
    emo: i
    for i, emo in enumerate(EMOTIONS)
}

NUM_LABELS = len(EMOTIONS)

print(EMOTIONS)

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []
valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    # -----------------------------------------
    # remove unsure
    # -----------------------------------------

    counts = {

        k: v
        for k, v
        in counts.items()

        if k != "unsure"
    }

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

# =========================================================
# REP PROCESSING
# =========================================================

def process_rep(x):

    x = x.float()

    # -----------------------------------------
    # pooling
    # -----------------------------------------

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    # -----------------------------------------
    # normalize
    # -----------------------------------------

    x = (
        x - x.mean()
    ) / (
        x.std() + 1e-6
    )

    return x

# =========================================================
# CACHE REPRESENTATIONS
# =========================================================

print("\nCaching representations...")

cached_audio = []
cached_llm = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    rep_id = row["id"]

    rep_path = os.path.join(
        REP_ROOT,
        f"{rep_id}.pt"
    )

    rep = torch.load(
        rep_path,
        map_location="cpu"
    )

    audio_x = process_rep(
        rep[AUDIO_LAYER]
    )

    llm_x = process_rep(
        rep[LLM_LAYER]
    )

    cached_audio.append(audio_x)
    cached_llm.append(llm_x)

cached_audio = torch.stack(cached_audio)
cached_llm = torch.stack(cached_llm)

cached_Y = torch.tensor(
    targets,
    dtype=torch.float
)

print("\nAudio shape:")
print(cached_audio.shape)

print("\nLLM shape:")
print(cached_llm.shape)

# =========================================================
# DATASET
# =========================================================

class FusionDataset(Dataset):

    def __init__(
        self,
        X,
        Y
    ):

        self.X = X
        self.Y = Y

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.Y[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# EVALUATION
# =========================================================

# =========================================================
# EVALUATION
# =========================================================

def evaluate(
    model,
    loader
):

    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            preds = (
                probs > THRESHOLD
            ).float()

            all_probs.append(
                probs.cpu().numpy()
            )

            all_preds.append(
                preds.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

    all_probs = np.concatenate(all_probs)

    all_preds = np.concatenate(all_preds)

    all_labels = np.concatenate(all_labels)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        all_labels,
        all_preds,
        average="micro",
        zero_division=0
    )

    return (
        macro_f1,
        micro_f1,
        all_probs,
        all_preds,
        all_labels
    )

# =========================================================
# RUN FUSION
# =========================================================

results = []

for alpha in ALPHAS:

    print("\n=================================")
    print(f"ALPHA = {alpha}")
    print("=================================")

    # -----------------------------------------------------
    # FUSION
    # -----------------------------------------------------

    fused_X = (

        alpha * cached_audio

        +

        (1 - alpha) * cached_llm
    )

    # -----------------------------------------------------
    # SPLIT
    # -----------------------------------------------------

    train_dataset = FusionDataset(

        fused_X[train_idx],
        cached_Y[train_idx]
    )

    val_dataset = FusionDataset(

        fused_X[val_idx],
        cached_Y[val_idx]
    )

    test_dataset = FusionDataset(

        fused_X[test_idx],
        cached_Y[test_idx]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # -----------------------------------------------------
    # MODEL
    # -----------------------------------------------------

    input_dim = fused_X.shape[-1]

    model = LinearProbe(
        input_dim,
        NUM_LABELS
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LR
    )

    best_val = -1

    # =====================================================
    # TRAIN
    # =====================================================

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        (
            val_macro,
            val_micro,
            _,
            _,
            _
        )  = evaluate(
            model,
            val_loader
        )

        print(
            f"Epoch {epoch} | "
            f"Loss {total_loss:.4f} | "
            f"Val Macro {val_macro:.4f} | "
            f"Val Micro {val_micro:.4f}"
        )

        if val_macro > best_val:

            best_val = val_macro

            torch.save(
                model.state_dict(),
                f"/content/fusion_alpha_{alpha}.pt"
            )

    # =====================================================
    # TEST
    # =====================================================

    model.load_state_dict(
        torch.load(
            f"/content/fusion_alpha_{alpha}.pt"
        )
    )

    (
      test_macro,
      test_micro,
      all_probs,
      all_preds,
      all_labels
    ) = evaluate(
      model,
      test_loader
    )
    # =====================================================
    # SAVE TEST PREDICTIONS
    # =====================================================

    os.makedirs(
        "/content/evaluation_predictions",
        exist_ok=True
    )

    eval_df = pd.DataFrame({

        "sample_index": test_idx,

        "sample_id": df.iloc[test_idx]["id"].values,

        "label_counts": df.iloc[test_idx]["label_counts"].values

    })

    for i, emo in enumerate(EMOTIONS):

        eval_df[f"{emo}_gold"] = all_labels[:, i]

        eval_df[f"{emo}_pred"] = all_preds[:, i]

        eval_df[f"{emo}_prob"] = all_probs[:, i]

    print("\nTEST RESULTS")
    print(
        f"Macro F1: {test_macro:.4f}"
    )
    print(
        f"Micro F1: {test_micro:.4f}"
    )

    results.append({

        "alpha": alpha,

        "macro_f1": test_macro,

        "micro_f1": test_micro
    })

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "/content/fusion_results.csv",
    index=False
)

print("\nFINAL RESULTS")
print(results_df)

In [ ]:

# =========================================================
# RESIDUAL FUSION
# ADD BACK SPEECH-CONDITIONED INFORMATION
# =========================================================

# ---------------------------------------------
# SMALL RESIDUAL INJECTION
# ---------------------------------------------

ALPHAS = [

    0.00,   # baseline

    0.01,

    0.03,

    0.07,

    0.10,

    0.20,

    0.50,

    1.2
]

# =========================================================
# RUN RESIDUAL FUSION
# =========================================================

results = []

for alpha in ALPHAS:

    print("\n=================================")
    print(f"ALPHA = {alpha}")
    print("=================================")

    # -----------------------------------------------------
    # RESIDUAL FUSION
    # -----------------------------------------------------

    fused_X = (

        cached_llm

        +

        alpha * cached_audio
    )

    # -----------------------------------------------------
    # DATASETS
    # -----------------------------------------------------

    train_dataset = FusionDataset(

        fused_X[train_idx],
        cached_Y[train_idx]
    )

    val_dataset = FusionDataset(

        fused_X[val_idx],
        cached_Y[val_idx]
    )

    test_dataset = FusionDataset(

        fused_X[test_idx],
        cached_Y[test_idx]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # -----------------------------------------------------
    # MODEL
    # -----------------------------------------------------

    input_dim = fused_X.shape[-1]

    model = LinearProbe(
        input_dim,
        NUM_LABELS
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LR
    )

    best_val = -1

    # =====================================================
    # TRAIN
    # =====================================================

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        for x, y in train_loader:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        (
            val_macro,
            val_micro,
            _,
            _,
            _
        )  = evaluate(
            model,
            val_loader
        )

        print(
            f"Epoch {epoch} | "
            f"Loss {total_loss:.4f} | "
            f"Val Macro {val_macro:.4f} | "
            f"Val Micro {val_micro:.4f}"
        )

        if val_macro > best_val:

            best_val = val_macro

            torch.save(
                model.state_dict(),
                f"/content/residual_alpha_{alpha}.pt"
            )

    # =====================================================
    # TEST
    # =====================================================

    model.load_state_dict(
        torch.load(
            f"/content/residual_alpha_{alpha}.pt"
        )
    )

    (
        test_macro,
        test_micro,
        all_probs,
        all_preds,
        all_labels
    ) = evaluate(
        model,
        test_loader
    )
    # =====================================================
    # SAVE TEST PREDICTIONS
    # =====================================================

    os.makedirs(
        "/content/evaluation_predictions",
        exist_ok=True
    )

    eval_df = pd.DataFrame({

        "sample_index": test_idx,

        "sample_id": df.iloc[test_idx]["id"].values,

        "label_counts": df.iloc[test_idx]["label_counts"].values

    })

    for i, emo in enumerate(EMOTIONS):

        eval_df[f"{emo}_gold"] = all_labels[:, i]

        eval_df[f"{emo}_pred"] = all_preds[:, i]

        eval_df[f"{emo}_prob"] = all_probs[:, i]

    print("\nTEST RESULTS")
    print(
        f"Macro F1: {test_macro:.4f}"
    )
    print(
        f"Micro F1: {test_micro:.4f}"
    )

    results.append({

        "alpha": alpha,

        "macro_f1": test_macro,

        "micro_f1": test_micro
    })
    print(alpha)
    eval_df.to_csv(

    f"/content/drive/MyDrive/evaluation_predictions/flamingo/fusion_alpha_{alpha:.2f}.csv",

    index=False
)

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    "/content/residual_fusion_results.csv",
    index=False
)

print("\nFINAL RESULTS")
print(results_df)
